# V4 Model Analysis: Why Don't Text Embeddings Help?

This notebook analyzes the V4 multi-label predictions to understand:
1. Which models correctly predict which user-subreddit pairs
2. How predictions overlap between models
3. Performance across user/subreddit popularity buckets
4. Fusion architecture diagnostics

**Models Compared:**
- Cosine Similarity (4096-dim text embeddings)
- NCF Baseline (Model B, no text)
- NCF + Text (Model C, with text embeddings)

## 1. Setup and Configuration

In [ ]:
# Install required packages
!pip install matplotlib-venn numpy pandas tqdm torch

In [ ]:
import numpy as np
import pandas as pd
import json
import os
from pathlib import Path
from collections import defaultdict
import matplotlib.pyplot as plt
from matplotlib_venn import venn3, venn3_circles
from tqdm import tqdm
import torch
import warnings
warnings.filterwarnings('ignore')

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

print("Setup complete!")

In [ ]:
# ============================================================
# CONFIGURATION - Update paths as needed
# ============================================================

BASE_DIR = Path('/content/drive/MyDrive/CIS 5300')

# Prediction files
PREDICTIONS_DIR = BASE_DIR / 'predictions'
PRED_FILES = {
    'cosine_4096': PREDICTIONS_DIR / 'cosine_v4_4096.npy',
    'ncf_baseline': PREDICTIONS_DIR / 'predictions_model_B.npy',
    'ncf_text': PREDICTIONS_DIR / 'predictions_model_C.npy',
}

# Ground truth
GROUND_TRUTH_FILE = PREDICTIONS_DIR / 'ground_truth_v4_test.npy'

# Data directories
DATA_DIR = BASE_DIR / 'engage_corpus_processed_v4'
NCF_DATA_DIR = DATA_DIR / 'ncf_data'

# Mapping files
USER_MAPPING_FILE = DATA_DIR / 'user_mapping.json'
SUBREDDIT_MAPPING_FILE = DATA_DIR / 'subreddit_mapping.json'

# Embedding directories
EMBEDDING_DIR = BASE_DIR / 'embeddings_qwen3_8b' / 'v4' / 'filtered' / 'test'

# Trained models (for fusion analysis)
MODEL_DIR = BASE_DIR / 'trained_models'
MODEL_C_PATH = MODEL_DIR / 'model_C_v4_text_best.pt'

# Popularity data (uploaded from local extraction)
POPULARITY_DIR = BASE_DIR / 'popularity_data_v4'

# Output directory for this analysis
OUTPUT_DIR = BASE_DIR / 'analysis_outputs_v4'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Top-K for evaluation
K = 10

print(f"Base directory: {BASE_DIR}")
print(f"Output directory: {OUTPUT_DIR}")

In [ ]:
# Verify files exist
print("Checking required files...\n")

all_files_exist = True
for name, path in PRED_FILES.items():
    exists = path.exists()
    status = "OK" if exists else "MISSING"
    print(f"  {name}: {status}")
    if not exists:
        all_files_exist = False

gt_exists = GROUND_TRUTH_FILE.exists()
print(f"  ground_truth: {'OK' if gt_exists else 'MISSING'}")

mapping_exists = USER_MAPPING_FILE.exists() and SUBREDDIT_MAPPING_FILE.exists()
print(f"  mappings: {'OK' if mapping_exists else 'MISSING'}")

# Check embedding index files
user_ids_exists = (EMBEDDING_DIR / 'user_ids.json').exists()
sub_names_exists = (EMBEDDING_DIR / 'subreddit_names.json').exists()
print(f"  user_ids.json: {'OK' if user_ids_exists else 'MISSING'}")
print(f"  subreddit_names.json: {'OK' if sub_names_exists else 'MISSING'}")

if all_files_exist and gt_exists and mapping_exists:
    print("\nAll required files found!")
else:
    print("\nWARNING: Some files are missing. Please check paths.")

## 2. Load Data and Create Index Mappings

**Important note**: The embedding arrays (.npy files) use a different index ordering than the NCF model indices.
- `user_ids.json`: List where position i contains the NCF user_id at embedding row i
- `subreddit_names.json`: List where position i contains the subreddit NAME at embedding row i

We must create reindex mappings to correctly access embeddings by NCF index.

In [ ]:
# Load ground truth
print("Loading ground truth...")
ground_truth = np.load(GROUND_TRUTH_FILE)
num_users, num_subreddits = ground_truth.shape
print(f"Ground truth shape: {ground_truth.shape}")
print(f"Total positive interactions: {int(ground_truth.sum())}")

# Load predictions
print("\nLoading predictions...")
predictions = {}
for name, path in PRED_FILES.items():
    print(f"  Loading {name}...")
    predictions[name] = np.load(path)
    print(f"    Shape: {predictions[name].shape}")

In [ ]:
# Load NCF mappings
print("Loading NCF mappings...")

with open(USER_MAPPING_FILE, 'r') as f:
    user_mapping = json.load(f)

with open(SUBREDDIT_MAPPING_FILE, 'r') as f:
    subreddit_mapping = json.load(f)

subreddit_names = subreddit_mapping['subreddits']
subreddit2idx = subreddit_mapping['subreddit2idx']
idx2subreddit = {v: k for k, v in subreddit2idx.items()}

print(f"Number of users (NCF): {user_mapping['num_users']}")
print(f"Number of subreddits (NCF): {subreddit_mapping['num_subreddits']}")
print(f"Sample subreddits: {subreddit_names[:10]}")

In [ ]:
# ============================================================
# Load embedding index files and create reindex mappings
# ============================================================

print("\n" + "="*70)
print("LOADING EMBEDDING INDEX FILES (FOR CORRECT ALIGNMENT)")
print("="*70)

# Load user_ids.json: List where position i = NCF user_id at embedding row i
user_ids_path = EMBEDDING_DIR / 'user_ids.json'
with open(user_ids_path, 'r') as f:
    embedding_user_ids = json.load(f)  # e.g., [0, 1, 2, 3, ...] or could be unsorted

# Load subreddit_names.json: List where position i = subreddit NAME at embedding row i
sub_names_path = EMBEDDING_DIR / 'subreddit_names.json'
with open(sub_names_path, 'r') as f:
    embedding_sub_names = json.load(f)  # e.g., ["AskReddit", "funny", ...]

print(f"\nEmbedding index files:")
print(f"  user_ids.json: {len(embedding_user_ids)} users")
print(f"    First 10: {embedding_user_ids[:10]}")
print(f"  subreddit_names.json: {len(embedding_sub_names)} subreddits")
print(f"    First 10: {embedding_sub_names[:10]}")

In [ ]:
# ============================================================
# Create reindex mappings: NCF index -> embedding row index
# ============================================================

print("\nCreating reindex mappings...")

# USER REINDEX: NCF user_id -> embedding row
# embedding_user_ids[row] = ncf_user_id at that row
# We need: given ncf_user_id, find the row
ncf_user_to_emb_row = {uid: row for row, uid in enumerate(embedding_user_ids)}

# Create reindex array for vectorized lookup
# user_reindex[ncf_user_id] = embedding_row (or -1 if missing)
user_reindex = np.full(num_users, -1, dtype=np.int32)
user_missing_mask = np.ones(num_users, dtype=bool)

for ncf_id in range(num_users):
    if ncf_id in ncf_user_to_emb_row:
        user_reindex[ncf_id] = ncf_user_to_emb_row[ncf_id]
        user_missing_mask[ncf_id] = False

print(f"\nUser reindexing:")
print(f"  NCF users: {num_users}")
print(f"  Embedding rows: {len(embedding_user_ids)}")
print(f"  Missing (no embedding): {user_missing_mask.sum()}")
print(f"  Coverage: {(~user_missing_mask).sum() / num_users * 100:.2f}%")

# SUBREDDIT REINDEX: NCF subreddit_idx -> embedding row
# embedding_sub_names[row] = subreddit_name at that row
# subreddit2idx[name] = ncf_idx
# We need: given ncf_idx, find the name, then find the row
sub_name_to_emb_row = {name: row for row, name in enumerate(embedding_sub_names)}

# Create reindex array
sub_reindex = np.full(num_subreddits, -1, dtype=np.int32)
sub_missing_mask = np.ones(num_subreddits, dtype=bool)

for ncf_idx in range(num_subreddits):
    sub_name = idx2subreddit.get(ncf_idx)
    if sub_name and sub_name in sub_name_to_emb_row:
        sub_reindex[ncf_idx] = sub_name_to_emb_row[sub_name]
        sub_missing_mask[ncf_idx] = False

print(f"\nSubreddit reindexing:")
print(f"  NCF subreddits: {num_subreddits}")
print(f"  Embedding rows: {len(embedding_sub_names)}")
print(f"  Missing (no embedding): {sub_missing_mask.sum()}")
print(f"  Coverage: {(~sub_missing_mask).sum() / num_subreddits * 100:.2f}%")

In [ ]:
# Verify reindexing is correct
print("\nVerifying reindex mappings...")

# Check a few examples
print("\nSample user mappings (NCF_idx -> emb_row):")
for ncf_id in [0, 1, 100, 1000]:
    if ncf_id < num_users:
        emb_row = user_reindex[ncf_id]
        # Verify: embedding_user_ids[emb_row] should equal ncf_id
        if emb_row >= 0:
            verified = embedding_user_ids[emb_row] == ncf_id
            print(f"  NCF user {ncf_id} -> emb row {emb_row} (verified: {verified})")
        else:
            print(f"  NCF user {ncf_id} -> MISSING")

print("\nSample subreddit mappings (NCF_idx -> emb_row):")
for ncf_idx in [0, 1, 100, 500]:
    if ncf_idx < num_subreddits:
        sub_name = idx2subreddit.get(ncf_idx, 'UNKNOWN')
        emb_row = sub_reindex[ncf_idx]
        # Verify: embedding_sub_names[emb_row] should equal sub_name
        if emb_row >= 0:
            verified = embedding_sub_names[emb_row] == sub_name
            print(f"  NCF sub {ncf_idx} ({sub_name}) -> emb row {emb_row} (verified: {verified})")
        else:
            print(f"  NCF sub {ncf_idx} ({sub_name}) -> MISSING")

In [ ]:
# Helper function to get embedding by NCF index (with reindexing)
def get_user_embedding(user_emb_array, ncf_user_id):
    """
    Get user embedding by NCF user index.
    Returns None if user has no embedding.
    """
    if ncf_user_id >= len(user_reindex):
        return None
    emb_row = user_reindex[ncf_user_id]
    if emb_row < 0 or emb_row >= user_emb_array.shape[0]:
        return None
    return user_emb_array[emb_row]

def get_sub_embedding(sub_emb_array, ncf_sub_idx):
    """
    Get subreddit embedding by NCF subreddit index.
    Returns None if subreddit has no embedding.
    """
    if ncf_sub_idx >= len(sub_reindex):
        return None
    emb_row = sub_reindex[ncf_sub_idx]
    if emb_row < 0 or emb_row >= sub_emb_array.shape[0]:
        return None
    return sub_emb_array[emb_row]

def get_user_embeddings_batch(user_emb_array, ncf_user_ids):
    """
    Get user embeddings for a batch of NCF user indices.
    Returns (embeddings, valid_mask) where valid_mask indicates which users have embeddings.
    """
    emb_rows = user_reindex[ncf_user_ids]
    valid_mask = (emb_rows >= 0) & (emb_rows < user_emb_array.shape[0])

    # Replace invalid indices with 0 for safe indexing (will be masked out)
    safe_rows = np.where(valid_mask, emb_rows, 0)
    embeddings = user_emb_array[safe_rows]

    return embeddings, valid_mask

def get_sub_embeddings_batch(sub_emb_array, ncf_sub_idxs):
    """
    Get subreddit embeddings for a batch of NCF subreddit indices.
    Returns (embeddings, valid_mask) where valid_mask indicates which subreddits have embeddings.
    """
    emb_rows = sub_reindex[ncf_sub_idxs]
    valid_mask = (emb_rows >= 0) & (emb_rows < sub_emb_array.shape[0])

    # Replace invalid indices with 0 for safe indexing (will be masked out)
    safe_rows = np.where(valid_mask, emb_rows, 0)
    embeddings = sub_emb_array[safe_rows]

    return embeddings, valid_mask

print("Helper functions defined for safe embedding access.")

## 3. Wrangle Data: Create Hit DataFrame

For each user-subreddit pair in the ground truth, determine whether each model correctly predicts it (i.e., the subreddit appears in the user's top-K predictions).

In [ ]:
def compute_hits_per_user_subreddit(predictions_dict, ground_truth, k=10):
    """
    For each positive (user, subreddit) pair in ground truth,
    check if each model correctly predicts it in top-K.

    Returns a DataFrame with columns:
    - user_id
    - subreddit_idx
    - subreddit_name
    - hit_<model_name> for each model (1 if hit, 0 if miss)
    """
    num_users, num_subreddits = ground_truth.shape

    # Pre-compute top-K indices for each model and user
    print("Computing top-K predictions for each model...")
    top_k_predictions = {}
    for model_name, preds in predictions_dict.items():
        print(f"  {model_name}...")
        # For each user, get top K subreddit indices
        top_k_predictions[model_name] = np.argsort(-preds, axis=1)[:, :k]

    # Collect hits for each positive pair
    print("\nComputing hits for each positive pair...")
    records = []

    # Find all positive pairs
    user_indices, sub_indices = np.where(ground_truth == 1)

    for i in tqdm(range(len(user_indices)), desc="Processing pairs"):
        user_id = user_indices[i]
        sub_idx = sub_indices[i]

        record = {
            'user_id': user_id,
            'subreddit_idx': sub_idx,
            'subreddit_name': idx2subreddit.get(sub_idx, f'unknown_{sub_idx}')
        }

        # Check hit for each model
        for model_name, top_k in top_k_predictions.items():
            hit = 1 if sub_idx in top_k[user_id] else 0
            record[f'hit_{model_name}'] = hit

        records.append(record)

    df = pd.DataFrame(records)
    return df, top_k_predictions

In [ ]:
# Create hits DataFrame
hits_df, top_k_preds = compute_hits_per_user_subreddit(predictions, ground_truth, k=K)

print(f"\nHits DataFrame shape: {hits_df.shape}")
print(f"\nColumns: {hits_df.columns.tolist()}")
print(f"\nSample rows:")
hits_df.head(10)

In [ ]:
# Summary statistics
print("Hit Rate Summary (at user-subreddit level):")
print("="*50)
for col in hits_df.columns:
    if col.startswith('hit_'):
        model_name = col.replace('hit_', '')
        hit_rate = hits_df[col].mean()
        print(f"  {model_name}: {hit_rate:.4f} ({hit_rate*100:.2f}%)")

In [ ]:
# Save hits DataFrame
hits_df.to_csv(OUTPUT_DIR / 'hits_dataframe_v4.csv', index=False)
print(f"Saved hits DataFrame to {OUTPUT_DIR / 'hits_dataframe_v4.csv'}")

## 4. Venn Diagram Analysis

Visualize overlap between which user-subreddit pairs each model correctly predicts.

In [ ]:
def create_venn_diagram(df, model_cols, title, figsize=(10, 8), save_path=None):
    """
    Create a Venn diagram showing overlap of hits between three models.
    """
    if len(model_cols) != 3:
        raise ValueError("Need exactly 3 models for Venn diagram")

    # Create sets of indices for each model's hits
    sets = []
    labels = []
    for col in model_cols:
        hit_indices = set(df[df[col] == 1].index.tolist())
        sets.append(hit_indices)
        model_name = col.replace('hit_', '').replace('_', ' ').title()
        labels.append(f"{model_name}\n({len(hit_indices):,})")

    A, B, C = sets

    Abc = len(A - B - C)
    aBc = len(B - A - C)
    abC = len(C - A - B)
    ABc = len((A & B) - C)
    AbC = len((A & C) - B)
    aBC = len((B & C) - A)
    ABC = len(A & B & C)

    fig, ax = plt.subplots(figsize=figsize)

    v = venn3(subsets=(Abc, aBc, ABc, abC, AbC, aBC, ABC),
              set_labels=labels,
              ax=ax)

    venn3_circles(subsets=(Abc, aBc, ABc, abC, AbC, aBC, ABC),
                  linestyle='solid', linewidth=1, ax=ax)

    plt.title(title, fontsize=14, fontweight='bold')

    total = len(df)
    total_hits = len(A | B | C)
    no_hits = total - total_hits
    plt.figtext(0.5, 0.02,
                f"Total pairs: {total:,} | Any hit: {total_hits:,} | No hits: {no_hits:,}",
                ha='center', fontsize=10)

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Saved to {save_path}")

    plt.show()

    stats = {
        'only_cosine': Abc,
        'only_ncf_baseline': aBc,
        'only_ncf_text': abC,
        'cosine_and_ncf_baseline': ABc,
        'cosine_and_ncf_text': AbC,
        'ncf_baseline_and_ncf_text': aBC,
        'all_three': ABC,
        'any_hit': total_hits,
        'no_hits': no_hits,
        'total': total
    }

    return stats

In [ ]:
# Create Venn Diagram
model_cols = ['hit_cosine_4096', 'hit_ncf_baseline', 'hit_ncf_text']
venn_stats = create_venn_diagram(
    hits_df,
    model_cols,
    title='V4 Model Hit Overlap (K=10)',
    save_path=OUTPUT_DIR / 'venn_diagram_v4.png'
)

print("\nVenn Diagram Statistics:")
for k, v in venn_stats.items():
    print(f"  {k}: {v:,}")

In [ ]:
# Print detailed statistics
print("Venn Diagram Statistics:")
print("="*50)
for key, value in venn_stats.items():
    pct = value / venn_stats['total'] * 100
    print(f"  {key}: {value:,} ({pct:.2f}%)")

# Key insights
print("\n" + "="*50)
print("KEY INSIGHTS:")
print("="*50)

# How much does NCF+Text add over NCF baseline?
ncf_text_unique = venn_stats['only_ncf_text'] + venn_stats['cosine_and_ncf_text']
ncf_baseline_unique = venn_stats['only_ncf_baseline'] + venn_stats['cosine_and_ncf_baseline']
print(f"\nPairs hit ONLY by NCF+Text (not NCF baseline): {venn_stats['only_ncf_text'] + venn_stats['cosine_and_ncf_text'] - venn_stats['cosine_and_ncf_baseline']:,}")
print(f"Pairs hit ONLY by NCF baseline (not NCF+Text): {venn_stats['only_ncf_baseline'] + venn_stats['cosine_and_ncf_baseline'] - venn_stats['cosine_and_ncf_text']:,}")

# Overlap between NCF models
ncf_overlap = venn_stats['ncf_baseline_and_ncf_text'] + venn_stats['all_three']
ncf_baseline_total = venn_stats['only_ncf_baseline'] + venn_stats['cosine_and_ncf_baseline'] + venn_stats['ncf_baseline_and_ncf_text'] + venn_stats['all_three']
ncf_text_total = venn_stats['only_ncf_text'] + venn_stats['cosine_and_ncf_text'] + venn_stats['ncf_baseline_and_ncf_text'] + venn_stats['all_three']

print(f"\nNCF Baseline total hits: {ncf_baseline_total:,}")
print(f"NCF+Text total hits: {ncf_text_total:,}")
print(f"Overlap between NCF models: {ncf_overlap:,} ({ncf_overlap/max(ncf_baseline_total, ncf_text_total)*100:.1f}% of larger)")

In [ ]:
# Save Venn statistics
with open(OUTPUT_DIR / 'venn_statistics.json', 'w') as f:
    json.dump(venn_stats, f, indent=2)
print(f"Saved Venn statistics to {OUTPUT_DIR / 'venn_statistics.json'}")

## 5. Load Popularity Data

Load the popularity metrics generated by the local extraction script.

In [ ]:
# Check if popularity data exists
popularity_exists = POPULARITY_DIR.exists()
print(f"Popularity data directory exists: {popularity_exists}")

if popularity_exists:
    print(f"\nFiles in {POPULARITY_DIR}:")
    for f in os.listdir(POPULARITY_DIR):
        print(f"  {f}")

In [ ]:
# Load popularity data (if available)
if popularity_exists:
    # Load user popularity
    with open(POPULARITY_DIR / 'user_popularity.json', 'r') as f:
        user_popularity = json.load(f)

    # Load subreddit popularity
    with open(POPULARITY_DIR / 'subreddit_popularity.json', 'r') as f:
        subreddit_popularity = json.load(f)

    print(f"Loaded user popularity for {len(user_popularity)} users")
    print(f"Loaded subreddit popularity for {len(subreddit_popularity)} subreddits")

    # Sample
    print("\nSample user popularity:")
    for uid in list(user_popularity.keys())[:3]:
        print(f"  User {uid}: {user_popularity[uid]}")

    print("\nSample subreddit popularity:")
    for sub in list(subreddit_popularity.keys())[:3]:
        print(f"  {sub}: {subreddit_popularity[sub]}")
else:
    print("\nPopularity data not found!")
    print("Please run the local extraction script and upload the results.")
    print(f"Expected location: {POPULARITY_DIR}")

    # Create placeholder data structure for testing
    print("\nCreating placeholder popularity data from test interactions...")

    # Count interactions per user and subreddit from ground truth
    user_popularity = {}
    subreddit_popularity = {}

    user_interaction_counts = ground_truth.sum(axis=1)
    sub_interaction_counts = ground_truth.sum(axis=0)

    for user_id in range(num_users):
        user_popularity[str(user_id)] = {
            'total_interactions': int(user_interaction_counts[user_id]),
            'num_subreddits': int((ground_truth[user_id] > 0).sum())
        }

    for sub_idx in range(num_subreddits):
        sub_name = idx2subreddit.get(sub_idx, f'unknown_{sub_idx}')
        subreddit_popularity[sub_name] = {
            'total_interactions': int(sub_interaction_counts[sub_idx]),
            'num_users': int((ground_truth[:, sub_idx] > 0).sum())
        }

    print(f"Created placeholder data for {len(user_popularity)} users and {len(subreddit_popularity)} subreddits")

In [ ]:
# Add popularity info to hits DataFrame
print("Adding popularity information to hits DataFrame...")

# User popularity (num_subreddits = diversity)
hits_df['user_num_subreddits'] = hits_df['user_id'].apply(
    lambda x: user_popularity.get(str(x), {}).get('num_subreddits', 0)
)
hits_df['user_total_interactions'] = hits_df['user_id'].apply(
    lambda x: user_popularity.get(str(x), {}).get('total_interactions', 0)
)

# Subreddit popularity
hits_df['subreddit_num_users'] = hits_df['subreddit_name'].apply(
    lambda x: subreddit_popularity.get(x, {}).get('num_users', 0)
)
hits_df['subreddit_total_interactions'] = hits_df['subreddit_name'].apply(
    lambda x: subreddit_popularity.get(x, {}).get('total_interactions', 0)
)

print(f"Updated DataFrame shape: {hits_df.shape}")
hits_df.head()

## 6. Bucket-Level Analysis

Analyze model performance across:
1. Subreddit popularity buckets
2. User diversity buckets (number of subreddits they interact with)

In [ ]:
def compute_metrics_for_subset(predictions_dict, ground_truth, user_mask=None, sub_mask=None, k=10):
    """
    Compute HR@K and NDCG@K for a subset of users/subreddits.

    Args:
        predictions_dict: Dict of model_name -> (num_users, num_subs) prediction matrix
        ground_truth: (num_users, num_subs) binary ground truth matrix
        user_mask: Boolean array of which users to include (None = all)
        sub_mask: Boolean array of which subreddits to include (None = all)
        k: Top-K for evaluation

    Returns:
        Dict of model_name -> {'hr@k': float, 'ndcg@k': float}
    """
    num_users, num_subs = ground_truth.shape

    if user_mask is None:
        user_mask = np.ones(num_users, dtype=bool)
    if sub_mask is None:
        sub_mask = np.ones(num_subs, dtype=bool)

    # Get indices
    user_indices = np.where(user_mask)[0]
    sub_indices = np.where(sub_mask)[0]

    results = {}

    for model_name, preds in predictions_dict.items():
        hr_sum = 0
        ndcg_sum = 0
        valid_users = 0

        for user_id in user_indices:
            # Get ground truth for this user, filtered to selected subreddits
            user_gt = ground_truth[user_id, sub_mask]

            # Skip users with no relevant items in this subset
            num_relevant = user_gt.sum()
            if num_relevant == 0:
                continue

            valid_users += 1

            # Get predictions for this user, filtered to selected subreddits
            user_preds = preds[user_id, sub_mask]

            # Get top K from the filtered predictions
            top_k_local_indices = np.argsort(-user_preds)[:k]

            # Check hits
            top_k_relevance = user_gt[top_k_local_indices]

            # HR@K
            if top_k_relevance.sum() > 0:
                hr_sum += 1

            # NDCG@K
            dcg = 0.0
            for rank, rel in enumerate(top_k_relevance, start=1):
                if rel > 0:
                    dcg += rel / np.log2(rank + 1)

            ideal_relevance = np.array([1] * min(int(num_relevant), k) + [0] * max(0, k - int(num_relevant)))
            idcg = sum(rel / np.log2(rank + 1) for rank, rel in enumerate(ideal_relevance, start=1) if rel > 0)

            if idcg > 0:
                ndcg_sum += dcg / idcg

        if valid_users > 0:
            results[model_name] = {
                'hr@k': hr_sum / valid_users,
                'ndcg@k': ndcg_sum / valid_users,
                'num_users': valid_users
            }
        else:
            results[model_name] = {
                'hr@k': 0.0,
                'ndcg@k': 0.0,
                'num_users': 0
            }

    return results

In [ ]:
# Create subreddit popularity buckets
print("Creating subreddit popularity buckets...")

# Get popularity for each subreddit index
sub_pop_array = np.zeros(num_subreddits)
for sub_idx in range(num_subreddits):
    sub_name = idx2subreddit.get(sub_idx, '')
    sub_pop_array[sub_idx] = subreddit_popularity.get(sub_name, {}).get('num_users', 0)

# Create quintile buckets
sub_percentiles = np.percentile(sub_pop_array[sub_pop_array > 0], [20, 40, 60, 80])
print(f"Subreddit popularity percentiles: {sub_percentiles}")

def get_sub_bucket(pop):
    if pop == 0:
        return 'Q0_zero'
    elif pop <= sub_percentiles[0]:
        return 'Q1_low'
    elif pop <= sub_percentiles[1]:
        return 'Q2_med_low'
    elif pop <= sub_percentiles[2]:
        return 'Q3_medium'
    elif pop <= sub_percentiles[3]:
        return 'Q4_med_high'
    else:
        return 'Q5_high'

sub_buckets = np.array([get_sub_bucket(p) for p in sub_pop_array])
print(f"Bucket distribution: {pd.Series(sub_buckets).value_counts().sort_index()}")

In [ ]:
# Compute metrics by subreddit popularity bucket
print("Computing metrics by subreddit popularity bucket...\n")

bucket_results_sub = []

for bucket_name in sorted(set(sub_buckets)):
    sub_mask = sub_buckets == bucket_name

    metrics = compute_metrics_for_subset(predictions, ground_truth, sub_mask=sub_mask, k=K)

    for model_name, model_metrics in metrics.items():
        bucket_results_sub.append({
            'bucket': bucket_name,
            'model': model_name,
            'hr@10': model_metrics['hr@k'],
            'ndcg@10': model_metrics['ndcg@k'],
            'num_users': model_metrics['num_users']
        })

bucket_df_sub = pd.DataFrame(bucket_results_sub)
print(bucket_df_sub.pivot(index='bucket', columns='model', values='hr@10'))

In [ ]:
# Create user diversity buckets
print("\nCreating user diversity buckets...")

# Get diversity for each user
user_diversity_array = np.zeros(num_users)
for user_id in range(num_users):
    user_diversity_array[user_id] = user_popularity.get(str(user_id), {}).get('num_subreddits', 0)

# Create quintile buckets
user_percentiles = np.percentile(user_diversity_array[user_diversity_array > 0], [20, 40, 60, 80])
print(f"User diversity percentiles: {user_percentiles}")

def get_user_bucket(div):
    if div == 0:
        return 'Q0_zero'
    elif div <= user_percentiles[0]:
        return 'Q1_low'
    elif div <= user_percentiles[1]:
        return 'Q2_med_low'
    elif div <= user_percentiles[2]:
        return 'Q3_medium'
    elif div <= user_percentiles[3]:
        return 'Q4_med_high'
    else:
        return 'Q5_high'

user_buckets = np.array([get_user_bucket(d) for d in user_diversity_array])
print(f"Bucket distribution: {pd.Series(user_buckets).value_counts().sort_index()}")

In [ ]:
# Compute metrics by user diversity bucket
print("Computing metrics by user diversity bucket...\n")

bucket_results_user = []

for bucket_name in sorted(set(user_buckets)):
    user_mask = user_buckets == bucket_name

    metrics = compute_metrics_for_subset(predictions, ground_truth, user_mask=user_mask, k=K)

    for model_name, model_metrics in metrics.items():
        bucket_results_user.append({
            'bucket': bucket_name,
            'model': model_name,
            'hr@10': model_metrics['hr@k'],
            'ndcg@10': model_metrics['ndcg@k'],
            'num_users': model_metrics['num_users']
        })

bucket_df_user = pd.DataFrame(bucket_results_user)
print(bucket_df_user.pivot(index='bucket', columns='model', values='hr@10'))

In [ ]:
# Plot bucket analysis
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Subreddit popularity - HR@10
ax = axes[0, 0]
pivot_sub_hr = bucket_df_sub.pivot(index='bucket', columns='model', values='hr@10')
pivot_sub_hr.plot(kind='bar', ax=ax, width=0.8)
ax.set_title('HR@10 by Subreddit Popularity', fontweight='bold')
ax.set_xlabel('Subreddit Popularity Bucket')
ax.set_ylabel('HR@10')
ax.legend(title='Model', bbox_to_anchor=(1.02, 1))
ax.tick_params(axis='x', rotation=45)

# Subreddit popularity - NDCG@10
ax = axes[0, 1]
pivot_sub_ndcg = bucket_df_sub.pivot(index='bucket', columns='model', values='ndcg@10')
pivot_sub_ndcg.plot(kind='bar', ax=ax, width=0.8)
ax.set_title('NDCG@10 by Subreddit Popularity', fontweight='bold')
ax.set_xlabel('Subreddit Popularity Bucket')
ax.set_ylabel('NDCG@10')
ax.legend(title='Model', bbox_to_anchor=(1.02, 1))
ax.tick_params(axis='x', rotation=45)

# User diversity - HR@10
ax = axes[1, 0]
pivot_user_hr = bucket_df_user.pivot(index='bucket', columns='model', values='hr@10')
pivot_user_hr.plot(kind='bar', ax=ax, width=0.8)
ax.set_title('HR@10 by User Diversity', fontweight='bold')
ax.set_xlabel('User Diversity Bucket')
ax.set_ylabel('HR@10')
ax.legend(title='Model', bbox_to_anchor=(1.02, 1))
ax.tick_params(axis='x', rotation=45)

# User diversity - NDCG@10
ax = axes[1, 1]
pivot_user_ndcg = bucket_df_user.pivot(index='bucket', columns='model', values='ndcg@10')
pivot_user_ndcg.plot(kind='bar', ax=ax, width=0.8)
ax.set_title('NDCG@10 by User Diversity', fontweight='bold')
ax.set_xlabel('User Diversity Bucket')
ax.set_ylabel('NDCG@10')
ax.legend(title='Model', bbox_to_anchor=(1.02, 1))
ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'bucket_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Saved to {OUTPUT_DIR / 'bucket_analysis.png'}")

In [ ]:
# Line plot version
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

bucket_order_sub = ['Q0_zero', 'Q1_low', 'Q2_med_low', 'Q3_medium', 'Q4_med_high', 'Q5_high']
bucket_order_user = ['Q0_zero', 'Q1_low', 'Q2_med_low', 'Q3_medium', 'Q4_med_high', 'Q5_high']

# Filter to existing buckets
bucket_order_sub = [b for b in bucket_order_sub if b in pivot_sub_hr.index]
bucket_order_user = [b for b in bucket_order_user if b in pivot_user_hr.index]

# Subreddit popularity - HR@10
ax = axes[0, 0]
for model in pivot_sub_hr.columns:
    values = [pivot_sub_hr.loc[b, model] if b in pivot_sub_hr.index else np.nan for b in bucket_order_sub]
    ax.plot(bucket_order_sub, values, marker='o', label=model, linewidth=2)
ax.set_title('HR@10 by Subreddit Popularity', fontweight='bold')
ax.set_xlabel('Subreddit Popularity Bucket')
ax.set_ylabel('HR@10')
ax.legend(title='Model')
ax.tick_params(axis='x', rotation=45)
ax.grid(True, alpha=0.3)

# Subreddit popularity - NDCG@10
ax = axes[0, 1]
for model in pivot_sub_ndcg.columns:
    values = [pivot_sub_ndcg.loc[b, model] if b in pivot_sub_ndcg.index else np.nan for b in bucket_order_sub]
    ax.plot(bucket_order_sub, values, marker='o', label=model, linewidth=2)
ax.set_title('NDCG@10 by Subreddit Popularity', fontweight='bold')
ax.set_xlabel('Subreddit Popularity Bucket')
ax.set_ylabel('NDCG@10')
ax.legend(title='Model')
ax.tick_params(axis='x', rotation=45)
ax.grid(True, alpha=0.3)

# User diversity - HR@10
ax = axes[1, 0]
for model in pivot_user_hr.columns:
    values = [pivot_user_hr.loc[b, model] if b in pivot_user_hr.index else np.nan for b in bucket_order_user]
    ax.plot(bucket_order_user, values, marker='o', label=model, linewidth=2)
ax.set_title('HR@10 by User Diversity', fontweight='bold')
ax.set_xlabel('User Diversity Bucket')
ax.set_ylabel('HR@10')
ax.legend(title='Model')
ax.tick_params(axis='x', rotation=45)
ax.grid(True, alpha=0.3)

# User diversity - NDCG@10
ax = axes[1, 1]
for model in pivot_user_ndcg.columns:
    values = [pivot_user_ndcg.loc[b, model] if b in pivot_user_ndcg.index else np.nan for b in bucket_order_user]
    ax.plot(bucket_order_user, values, marker='o', label=model, linewidth=2)
ax.set_title('NDCG@10 by User Diversity', fontweight='bold')
ax.set_xlabel('User Diversity Bucket')
ax.set_ylabel('NDCG@10')
ax.legend(title='Model')
ax.tick_params(axis='x', rotation=45)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'bucket_analysis_lines.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Saved to {OUTPUT_DIR / 'bucket_analysis_lines.png'}")

In [ ]:
# Save bucket analysis tables
bucket_df_sub.to_csv(OUTPUT_DIR / 'bucket_analysis_subreddit_popularity.csv', index=False)
bucket_df_user.to_csv(OUTPUT_DIR / 'bucket_analysis_user_diversity.csv', index=False)

print(f"Saved bucket analysis to:")
print(f"  {OUTPUT_DIR / 'bucket_analysis_subreddit_popularity.csv'}")
print(f"  {OUTPUT_DIR / 'bucket_analysis_user_diversity.csv'}")

## 7. Fusion Architecture Diagnostics

Analyze the NCF+Text model to check if the text projection layer is learning.

In [ ]:
# Load Model C (NCF + Text)
print("Loading Model C (NCF + Text) for fusion analysis...")

if MODEL_C_PATH.exists():
    checkpoint = torch.load(MODEL_C_PATH, map_location='cpu')

    if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
        state_dict = checkpoint['model_state_dict']
        print(f"Loaded from epoch {checkpoint.get('epoch', '?')}")
        print(f"Validation loss: {checkpoint.get('val_loss', '?')}")
    else:
        state_dict = checkpoint

    print(f"\nModel parameters:")
    for name, param in state_dict.items():
        print(f"  {name}: {param.shape}")
else:
    print(f"Model file not found: {MODEL_C_PATH}")
    state_dict = None

In [ ]:
# Analyze projection layer weights
if state_dict is not None:
    print("\n" + "="*60)
    print("PROJECTION LAYER ANALYSIS")
    print("="*60)

    # User text projection
    if 'user_text_proj.weight' in state_dict:
        user_proj_weight = state_dict['user_text_proj.weight'].numpy()
        user_proj_bias = state_dict.get('user_text_proj.bias', torch.zeros(user_proj_weight.shape[0])).numpy()

        print("\nUser Text Projection Layer:")
        print(f"  Weight shape: {user_proj_weight.shape}")
        print(f"  Weight mean: {user_proj_weight.mean():.6f}")
        print(f"  Weight std: {user_proj_weight.std():.6f}")
        print(f"  Weight min: {user_proj_weight.min():.6f}")
        print(f"  Weight max: {user_proj_weight.max():.6f}")
        print(f"  Bias mean: {user_proj_bias.mean():.6f}")
        print(f"  Bias std: {user_proj_bias.std():.6f}")

        # Compare to random initialization (Xavier uniform)
        # For linear layer, Xavier uniform: std = sqrt(2 / (fan_in + fan_out))
        fan_in, fan_out = user_proj_weight.shape[1], user_proj_weight.shape[0]
        xavier_std = np.sqrt(2.0 / (fan_in + fan_out))
        print(f"\n  Expected Xavier init std: {xavier_std:.6f}")
        print(f"  Actual std / Xavier std: {user_proj_weight.std() / xavier_std:.2f}x")

        if abs(user_proj_weight.std() - xavier_std) / xavier_std < 0.2:
            print("  WARNING: Weight std is close to initialization - layer may not be learning!")
        else:
            print("  OK: Weight std differs significantly from initialization")

    # Subreddit text projection
    if 'sub_text_proj.weight' in state_dict:
        sub_proj_weight = state_dict['sub_text_proj.weight'].numpy()
        sub_proj_bias = state_dict.get('sub_text_proj.bias', torch.zeros(sub_proj_weight.shape[0])).numpy()

        print("\nSubreddit Text Projection Layer:")
        print(f"  Weight shape: {sub_proj_weight.shape}")
        print(f"  Weight mean: {sub_proj_weight.mean():.6f}")
        print(f"  Weight std: {sub_proj_weight.std():.6f}")
        print(f"  Weight min: {sub_proj_weight.min():.6f}")
        print(f"  Weight max: {sub_proj_weight.max():.6f}")

        fan_in, fan_out = sub_proj_weight.shape[1], sub_proj_weight.shape[0]
        xavier_std = np.sqrt(2.0 / (fan_in + fan_out))
        print(f"\n  Expected Xavier init std: {xavier_std:.6f}")
        print(f"  Actual std / Xavier std: {sub_proj_weight.std() / xavier_std:.2f}x")

In [ ]:
# Embedding magnitude analysis
print("\n" + "="*60)
print("EMBEDDING MAGNITUDE ANALYSIS")
print("="*60)

# Load text embeddings
text_emb_dir = BASE_DIR / 'embeddings_qwen3_8b' / 'v4' / 'filtered' / 'test'

if (text_emb_dir / 'user_embeddings_128.npy').exists():
    user_text_emb = np.load(text_emb_dir / 'user_embeddings_128.npy')
    sub_text_emb = np.load(text_emb_dir / 'subreddit_embeddings_128.npy')

    print("\nText Embeddings (128-dim):")
    print(f"  User embeddings shape: {user_text_emb.shape}")
    print(f"  Subreddit embeddings shape: {sub_text_emb.shape}")

    # Handle NaN values properly when computing norms
    # Replace NaN with 0 for norm computation, then filter out those rows
    user_text_clean = np.nan_to_num(user_text_emb, nan=0.0)
    nan_mask = np.any(np.isnan(user_text_emb), axis=1)

    # Compute norms
    user_text_norms = np.linalg.norm(user_text_clean, axis=1)
    sub_text_norms = np.linalg.norm(sub_text_emb, axis=1)

    # For valid users only (excluding NaN rows)
    user_text_norms_valid = user_text_norms[~nan_mask]

    print(f"\n  User text embedding norms (excluding {nan_mask.sum()} NaN rows):")
    print(f"    Mean: {user_text_norms_valid.mean():.4f}")
    print(f"    Std: {user_text_norms_valid.std():.4f}")
    print(f"    Min: {user_text_norms_valid.min():.4f}")
    print(f"    Max: {user_text_norms_valid.max():.4f}")

    print(f"\n  Subreddit text embedding norms:")
    print(f"    Mean: {sub_text_norms.mean():.4f}")
    print(f"    Std: {sub_text_norms.std():.4f}")
    print(f"    Min: {sub_text_norms.min():.4f}")
    print(f"    Max: {sub_text_norms.max():.4f}")

    # IMPORTANT INSIGHT: Check if embeddings are L2-normalized
    if np.allclose(sub_text_norms, 1.0, atol=0.01):
        print("\n  NOTE: Text embeddings appear to be L2-NORMALIZED (norm ≈ 1.0)")
        print("     This explains why 'Before' histogram is not visible - all values are ~1.0")
        print("     The embeddings were likely normalized during preprocessing.")

    # Check for zero/near-zero embeddings
    user_zero_count = (user_text_norms < 0.01).sum()
    sub_zero_count = (sub_text_norms < 0.01).sum()
    print(f"\n  Users with near-zero embeddings: {user_zero_count} ({user_zero_count/len(user_text_norms)*100:.2f}%)")
    print(f"  Subreddits with near-zero embeddings: {sub_zero_count} ({sub_zero_count/len(sub_text_norms)*100:.2f}%)")
else:
    print("Text embeddings not found!")
    user_text_norms = None
    sub_text_norms = None

In [ ]:
# Compare NCF learned embeddings vs text embeddings
if state_dict is not None:
    print("\n" + "="*60)
    print("NCF vs TEXT EMBEDDING COMPARISON")
    print("="*60)

    # Get NCF embeddings
    user_ncf_gmf = state_dict.get('user_embedding_gmf.weight', None)
    sub_ncf_gmf = state_dict.get('sub_embedding_gmf.weight', None)

    if user_ncf_gmf is not None:
        user_ncf_gmf = user_ncf_gmf.numpy()
        user_ncf_norms = np.linalg.norm(user_ncf_gmf, axis=1)

        print("\nUser NCF GMF Embeddings (64-dim):")
        print(f"  Shape: {user_ncf_gmf.shape}")
        print(f"  Norm mean: {user_ncf_norms.mean():.4f}")
        print(f"  Norm std: {user_ncf_norms.std():.4f}")

    if sub_ncf_gmf is not None:
        sub_ncf_gmf = sub_ncf_gmf.numpy()
        sub_ncf_norms = np.linalg.norm(sub_ncf_gmf, axis=1)

        print("\nSubreddit NCF GMF Embeddings (64-dim):")
        print(f"  Shape: {sub_ncf_gmf.shape}")
        print(f"  Norm mean: {sub_ncf_norms.mean():.4f}")
        print(f"  Norm std: {sub_ncf_norms.std():.4f}")

    # Compare scales
    if user_text_norms is not None and user_ncf_gmf is not None:
        print("\nScale Comparison:")
        print(f"  User text norm mean / NCF norm mean: {user_text_norms.mean() / user_ncf_norms.mean():.2f}x")
        print(f"  Sub text norm mean / NCF norm mean: {sub_text_norms.mean() / sub_ncf_norms.mean():.2f}x")

        if abs(user_text_norms.mean() / user_ncf_norms.mean() - 1) > 0.5:
            print("  WARNING: Significant scale mismatch between text and NCF embeddings!")

In [ ]:
# Plot embedding norm distributions
if user_text_norms is not None and state_dict is not None:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # User embeddings
    ax = axes[0]
    # Note: Text embeddings are L2-normalized (all norms ≈ 1.0), so they won't show well
    # We'll still plot them but with a note
    valid_user_norms = user_text_norms[user_text_norms > 0.01]  # Exclude near-zero
    ax.hist(valid_user_norms, bins=50, alpha=0.7, label='Text (128-dim)', density=True)
    ax.hist(user_ncf_norms, bins=50, alpha=0.7, label='NCF GMF (64-dim)', density=True)
    ax.set_xlabel('Embedding Norm')
    ax.set_ylabel('Density')
    ax.set_title('User Embedding Norm Distributions')
    ax.legend()

    # Add note if text embeddings are normalized
    if np.allclose(valid_user_norms, 1.0, atol=0.05):
        ax.annotate('Text embeddings are L2-normalized\n(all norms ≈ 1.0)',
                    xy=(0.95, 0.95), xycoords='axes fraction',
                    ha='right', va='top', fontsize=8,
                    bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.5))

    # Subreddit embeddings
    ax = axes[1]
    ax.hist(sub_text_norms, bins=50, alpha=0.7, label='Text (128-dim)', density=True)
    ax.hist(sub_ncf_norms, bins=50, alpha=0.7, label='NCF GMF (64-dim)', density=True)
    ax.set_xlabel('Embedding Norm')
    ax.set_ylabel('Density')
    ax.set_title('Subreddit Embedding Norm Distributions')
    ax.legend()

    # Add note if text embeddings are normalized
    if np.allclose(sub_text_norms, 1.0, atol=0.05):
        ax.annotate('Text embeddings are L2-normalized\n(all norms ≈ 1.0)',
                    xy=(0.95, 0.95), xycoords='axes fraction',
                    ha='right', va='top', fontsize=8,
                    bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.5))

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'embedding_norm_distributions.png', dpi=150, bbox_inches='tight')
    plt.show()

    print("\nNOTE: If 'Text' histogram is not visible, it's because the text embeddings")
    print("   are L2-normalized (all norms = 1.0), forming a spike that may be off-scale.")
    print("   This is a preprocessing choice, not a bug.")

## 8. Grouped Venn Diagrams (Recall/Precision)

Create Venn diagrams based on model performance thresholds for users and subreddits.

In [ ]:
# ============================================================
# THRESHOLD CONFIGURATION - Adjust these values and re-run
# ============================================================

# Recall threshold: What fraction of a user's/subreddit's relevant items
# must be in top-K to consider the model "good" for that entity?
RECALL_THRESHOLD = 0.5  # 50% of relevant items in top-K

# Precision threshold: What fraction of a model's top-K predictions
# for a user must be relevant to consider the model "good"?
PRECISION_THRESHOLD = 0.1  # 10% of top-K are relevant (1 out of 10)

print(f"Recall threshold: {RECALL_THRESHOLD}")
print(f"Precision threshold: {PRECISION_THRESHOLD}")

In [ ]:
def compute_user_recall_per_model(predictions_dict, ground_truth, k=10):
    """
    For each user, compute recall@K for each model.
    Recall = (# relevant items in top-K) / (# total relevant items)
    """
    num_users = ground_truth.shape[0]

    user_recall = {model: np.zeros(num_users) for model in predictions_dict.keys()}
    user_num_relevant = np.zeros(num_users)

    for user_id in range(num_users):
        num_relevant = ground_truth[user_id].sum()
        user_num_relevant[user_id] = num_relevant

        if num_relevant == 0:
            continue

        for model_name, preds in predictions_dict.items():
            top_k = np.argsort(-preds[user_id])[:k]
            hits = ground_truth[user_id, top_k].sum()
            user_recall[model_name][user_id] = hits / num_relevant

    return user_recall, user_num_relevant


def compute_user_precision_per_model(predictions_dict, ground_truth, k=10):
    """
    For each user, compute precision@K for each model.
    Precision = (# relevant items in top-K) / K
    """
    num_users = ground_truth.shape[0]

    user_precision = {model: np.zeros(num_users) for model in predictions_dict.keys()}

    for user_id in range(num_users):
        for model_name, preds in predictions_dict.items():
            top_k = np.argsort(-preds[user_id])[:k]
            hits = ground_truth[user_id, top_k].sum()
            user_precision[model_name][user_id] = hits / k

    return user_precision

In [ ]:
# Compute recall and precision for each user
print("Computing per-user recall and precision...")

user_recall, user_num_relevant = compute_user_recall_per_model(predictions, ground_truth, k=K)
user_precision = compute_user_precision_per_model(predictions, ground_truth, k=K)

# Only consider users with at least 1 relevant item
valid_users = user_num_relevant > 0
print(f"Users with at least 1 relevant item: {valid_users.sum():,}")

In [ ]:
# Create Recall Venn Diagram for Users
print(f"\nCreating Recall Venn Diagram (threshold = {RECALL_THRESHOLD})...")

# Categorize users based on recall threshold
user_good_recall = {}
for model_name in predictions.keys():
    user_good_recall[model_name] = set(
        np.where((user_recall[model_name] >= RECALL_THRESHOLD) & valid_users)[0]
    )
    print(f"  {model_name}: {len(user_good_recall[model_name]):,} users with recall >= {RECALL_THRESHOLD}")

In [ ]:
# Plot Recall Venn Diagram
fig, ax = plt.subplots(figsize=(10, 8))

A = user_good_recall['cosine_4096']
B = user_good_recall['ncf_baseline']
C = user_good_recall['ncf_text']

Abc = len(A - B - C)
aBc = len(B - A - C)
abC = len(C - A - B)
ABc = len((A & B) - C)
AbC = len((A & C) - B)
aBC = len((B & C) - A)
ABC = len(A & B & C)

v = venn3(subsets=(Abc, aBc, ABc, abC, AbC, aBC, ABC),
          set_labels=[f'Cosine 4096\n({len(A):,})',
                      f'NCF Baseline\n({len(B):,})',
                      f'NCF+Text\n({len(C):,})'],
          ax=ax)

venn3_circles(subsets=(Abc, aBc, ABc, abC, AbC, aBC, ABC),
              linestyle='solid', linewidth=1, ax=ax)

plt.title(f'User Recall Venn Diagram\n(Recall >= {RECALL_THRESHOLD})',
          fontsize=14, fontweight='bold')

total_valid = valid_users.sum()
good_any = len(A | B | C)
plt.figtext(0.5, 0.02,
            f"Total valid users: {total_valid:,} | Good recall (any): {good_any:,} | None: {total_valid - good_any:,}",
            ha='center', fontsize=10)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'venn_user_recall.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Create Precision Venn Diagram for Users
print(f"\nCreating Precision Venn Diagram (threshold = {PRECISION_THRESHOLD})...")

user_good_precision = {}
for model_name in predictions.keys():
    user_good_precision[model_name] = set(
        np.where((user_precision[model_name] >= PRECISION_THRESHOLD) & valid_users)[0]
    )
    print(f"  {model_name}: {len(user_good_precision[model_name]):,} users with precision >= {PRECISION_THRESHOLD}")

In [ ]:
# Plot Precision Venn Diagram
fig, ax = plt.subplots(figsize=(10, 8))

A = user_good_precision['cosine_4096']
B = user_good_precision['ncf_baseline']
C = user_good_precision['ncf_text']

Abc = len(A - B - C)
aBc = len(B - A - C)
abC = len(C - A - B)
ABc = len((A & B) - C)
AbC = len((A & C) - B)
aBC = len((B & C) - A)
ABC = len(A & B & C)

v = venn3(subsets=(Abc, aBc, ABc, abC, AbC, aBC, ABC),
          set_labels=[f'Cosine 4096\n({len(A):,})',
                      f'NCF Baseline\n({len(B):,})',
                      f'NCF+Text\n({len(C):,})'],
          ax=ax)

venn3_circles(subsets=(Abc, aBc, ABc, abC, AbC, aBC, ABC),
              linestyle='solid', linewidth=1, ax=ax)

plt.title(f'User Precision Venn Diagram\n(Precision >= {PRECISION_THRESHOLD})',
          fontsize=14, fontweight='bold')

good_any = len(A | B | C)
plt.figtext(0.5, 0.02,
            f"Total valid users: {total_valid:,} | Good precision (any): {good_any:,} | None: {total_valid - good_any:,}",
            ha='center', fontsize=10)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'venn_user_precision.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Subreddit-level Venn diagrams
print("\nComputing per-subreddit recall...")

def compute_subreddit_recall_per_model(predictions_dict, ground_truth, k=10):
    """
    For each subreddit, compute "recall" = fraction of users who should
    engage with it that have it in their top-K.
    """
    num_users, num_subs = ground_truth.shape

    sub_recall = {model: np.zeros(num_subs) for model in predictions_dict.keys()}
    sub_num_relevant = np.zeros(num_subs)

    # Pre-compute top-K for each model
    top_k_preds = {}
    for model_name, preds in predictions_dict.items():
        top_k_preds[model_name] = np.argsort(-preds, axis=1)[:, :k]

    for sub_idx in tqdm(range(num_subs), desc="Processing subreddits"):
        # Users who should engage with this subreddit
        relevant_users = np.where(ground_truth[:, sub_idx] > 0)[0]
        num_relevant = len(relevant_users)
        sub_num_relevant[sub_idx] = num_relevant

        if num_relevant == 0:
            continue

        for model_name in predictions_dict.keys():
            # How many of these users have this subreddit in their top-K?
            hits = 0
            for user_id in relevant_users:
                if sub_idx in top_k_preds[model_name][user_id]:
                    hits += 1

            sub_recall[model_name][sub_idx] = hits / num_relevant

    return sub_recall, sub_num_relevant

sub_recall, sub_num_relevant = compute_subreddit_recall_per_model(predictions, ground_truth, k=K)

In [ ]:
# Subreddit Recall Venn Diagram
print(f"\nCreating Subreddit Recall Venn Diagram (threshold = {RECALL_THRESHOLD})...")

valid_subs = sub_num_relevant > 0

sub_good_recall = {}
for model_name in predictions.keys():
    sub_good_recall[model_name] = set(
        np.where((sub_recall[model_name] >= RECALL_THRESHOLD) & valid_subs)[0]
    )
    print(f"  {model_name}: {len(sub_good_recall[model_name]):,} subreddits with recall >= {RECALL_THRESHOLD}")

In [ ]:
# Plot Subreddit Recall Venn Diagram
fig, ax = plt.subplots(figsize=(10, 8))

A = sub_good_recall['cosine_4096']
B = sub_good_recall['ncf_baseline']
C = sub_good_recall['ncf_text']

Abc = len(A - B - C)
aBc = len(B - A - C)
abC = len(C - A - B)
ABc = len((A & B) - C)
AbC = len((A & C) - B)
aBC = len((B & C) - A)
ABC = len(A & B & C)

v = venn3(subsets=(Abc, aBc, ABc, abC, AbC, aBC, ABC),
          set_labels=[f'Cosine 4096\n({len(A):,})',
                      f'NCF Baseline\n({len(B):,})',
                      f'NCF+Text\n({len(C):,})'],
          ax=ax)

venn3_circles(subsets=(Abc, aBc, ABc, abC, AbC, aBC, ABC),
              linestyle='solid', linewidth=1, ax=ax)

plt.title(f'Subreddit Recall Venn Diagram\n(Recall >= {RECALL_THRESHOLD})',
          fontsize=14, fontweight='bold')

total_valid_subs = valid_subs.sum()
good_any = len(A | B | C)
plt.figtext(0.5, 0.02,
            f"Total valid subreddits: {total_valid_subs:,} | Good recall (any): {good_any:,}",
            ha='center', fontsize=10)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'venn_subreddit_recall.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Export subreddit names for each Venn region
print("\nExporting subreddit names for each Venn region...")

A = sub_good_recall['cosine_4096']
B = sub_good_recall['ncf_baseline']
C = sub_good_recall['ncf_text']

venn_regions = {
    'only_cosine': A - B - C,
    'only_ncf_baseline': B - A - C,
    'only_ncf_text': C - A - B,
    'cosine_and_ncf_baseline': (A & B) - C,
    'cosine_and_ncf_text': (A & C) - B,
    'ncf_baseline_and_ncf_text': (B & C) - A,
    'all_three': A & B & C,
}

subreddit_names_by_region = {}
for region_name, indices in venn_regions.items():
    names = [idx2subreddit.get(idx, f'unknown_{idx}') for idx in sorted(indices)]
    subreddit_names_by_region[region_name] = names
    print(f"  {region_name}: {len(names)} subreddits")

# Save to JSON
with open(OUTPUT_DIR / 'subreddit_venn_regions_recall.json', 'w') as f:
    json.dump(subreddit_names_by_region, f, indent=2)

print(f"\nSaved to {OUTPUT_DIR / 'subreddit_venn_regions_recall.json'}")

# Create detailed CSV export with recall values for each subreddit
print("\nCreating detailed export with recall values...")

detailed_records = []
for region_name, indices in venn_regions.items():
    for idx in sorted(indices):
        sub_name = idx2subreddit.get(idx, f'unknown_{idx}')
        detailed_records.append({
            'subreddit_idx': idx,
            'subreddit_name': sub_name,
            'venn_region': region_name,
            'recall_cosine_4096': sub_recall['cosine_4096'][idx],
            'recall_ncf_baseline': sub_recall['ncf_baseline'][idx],
            'recall_ncf_text': sub_recall['ncf_text'][idx],
            'num_relevant_users': int(sub_num_relevant[idx]),
            'recall_threshold': RECALL_THRESHOLD
        })

detailed_venn_df = pd.DataFrame(detailed_records)
detailed_venn_df.to_csv(OUTPUT_DIR / 'subreddit_venn_regions_recall_detailed.csv', index=False)
print(f"Saved detailed export to {OUTPUT_DIR / 'subreddit_venn_regions_recall_detailed.csv'}")
print(f"Total subreddits in Venn regions: {len(detailed_records)}")

In [ ]:
# Show sample subreddits from key regions
print("\n" + "="*60)
print("SAMPLE SUBREDDITS FROM KEY REGIONS")
print("="*60)

for region_name in ['only_cosine', 'only_ncf_baseline', 'only_ncf_text', 'all_three']:
    names = subreddit_names_by_region[region_name]
    print(f"\n{region_name} ({len(names)} total):")
    print(f"  Sample: {names[:20]}")

## 10. Deep Embedding Analysis

This section analyzes the text embeddings and their relationship to NCF embeddings. (This part uses proper reindex mappings for the embeddings)

In [ ]:
# Initialize diagnostics dictionary
embedding_diagnostics = {}

In [ ]:
# Load Model C (NCF + Text)
print("Loading Model C (NCF + Text) for fusion analysis...")

if MODEL_C_PATH.exists():
    checkpoint = torch.load(MODEL_C_PATH, map_location='cpu')

    if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
        state_dict = checkpoint['model_state_dict']
        print(f"Loaded from epoch {checkpoint.get('epoch', '?')}")
        print(f"Validation loss: {checkpoint.get('val_loss', '?')}")
    else:
        state_dict = checkpoint

    print(f"\nModel parameters:")
    for name, param in state_dict.items():
        print(f"  {name}: {param.shape}")
else:
    print(f"Model file not found: {MODEL_C_PATH}")
    state_dict = None

In [ ]:
# Load text embeddings
print("\n" + "="*60)
print("LOADING TEXT EMBEDDINGS")
print("="*60)

user_text_emb_path = EMBEDDING_DIR / 'user_embeddings_128.npy'
sub_text_emb_path = EMBEDDING_DIR / 'subreddit_embeddings_128.npy'

if user_text_emb_path.exists() and sub_text_emb_path.exists():
    user_text_emb = np.load(user_text_emb_path)
    sub_text_emb = np.load(sub_text_emb_path)

    print(f"\nText Embeddings (128-dim):")
    print(f"  User embeddings shape: {user_text_emb.shape}")
    print(f"  Subreddit embeddings shape: {sub_text_emb.shape}")

    # Check for NaN
    user_nan_rows = np.isnan(user_text_emb).any(axis=1).sum()
    sub_nan_rows = np.isnan(sub_text_emb).any(axis=1).sum()

    print(f"\n  User NaN rows: {user_nan_rows}")
    print(f"  Subreddit NaN rows: {sub_nan_rows}")

    embedding_diagnostics['user_text'] = {
        'shape': list(user_text_emb.shape),
        'nan_rows': int(user_nan_rows)
    }
    embedding_diagnostics['subreddit_text'] = {
        'shape': list(sub_text_emb.shape),
        'nan_rows': int(sub_nan_rows)
    }
else:
    print("Text embeddings not found!")
    user_text_emb = None
    sub_text_emb = None

In [ ]:
# ============================================================
# 10.4 TEXT SIMILARITY ANALYSIS FOR CO-ENGAGED PAIRS
# ============================================================

print("\n" + "="*70)
print("TEXT SIMILARITY FOR CO-ENGAGED USER-SUBREDDIT PAIRS")
print("="*70)

if user_text_emb is not None and sub_text_emb is not None:
    # Sample positive pairs from ground truth
    positive_users, positive_subs = np.where(ground_truth > 0)
    num_positive = len(positive_users)

    print(f"\nTotal positive pairs: {num_positive:,}")

    # Sample for efficiency
    sample_size = min(50000, num_positive)
    sample_indices = np.random.choice(num_positive, sample_size, replace=False)

    print(f"Sampling {sample_size:,} pairs for analysis...")

    # Clean user embeddings (replace NaN with zeros)
    user_text_clean = np.nan_to_num(user_text_emb, nan=0.0)

    # Normalize embeddings for cosine similarity
    user_norms = np.linalg.norm(user_text_clean, axis=1, keepdims=True)
    user_norms = np.where(user_norms > 0, user_norms, 1.0)  # Avoid division by zero
    user_text_normalized = user_text_clean / user_norms

    sub_norms = np.linalg.norm(sub_text_emb, axis=1, keepdims=True)
    sub_norms = np.where(sub_norms > 0, sub_norms, 1.0)
    sub_text_normalized = sub_text_emb / sub_norms

    # Compute cosine similarities for positive pairs WITH PROPER REINDEXING
    positive_similarities = []
    valid_pairs = 0
    skipped_user = 0
    skipped_sub = 0

    for idx in tqdm(sample_indices, desc="Computing positive pair similarities"):
        ncf_user_id = positive_users[idx]
        ncf_sub_idx = positive_subs[idx]

        # Use reindex mapping to get embedding row
        user_emb_row = user_reindex[ncf_user_id]
        sub_emb_row = sub_reindex[ncf_sub_idx]

        # Skip if no embedding available
        if user_emb_row < 0:
            skipped_user += 1
            continue
        if sub_emb_row < 0:
            skipped_sub += 1
            continue

        # Check bounds (should be fine if reindex is correct)
        if user_emb_row >= user_text_normalized.shape[0] or sub_emb_row >= sub_text_normalized.shape[0]:
            continue

        # Compute similarity using embedding rows
        sim = np.dot(user_text_normalized[user_emb_row], sub_text_normalized[sub_emb_row])
        if not np.isnan(sim):
            positive_similarities.append(sim)
            valid_pairs += 1

    positive_similarities = np.array(positive_similarities)

    print(f"\nPositive pairs analysis:")
    print(f"  Valid pairs computed: {valid_pairs:,}")
    print(f"  Skipped (no user embedding): {skipped_user:,}")
    print(f"  Skipped (no subreddit embedding): {skipped_sub:,}")

    if len(positive_similarities) > 0:
        print(f"\nPositive pair similarity stats:")
        print(f"  Mean: {positive_similarities.mean():.4f}")
        print(f"  Std: {positive_similarities.std():.4f}")
        print(f"  Min: {positive_similarities.min():.4f}")
        print(f"  Max: {positive_similarities.max():.4f}")

    # Generate random pairs for comparison (also with proper reindexing)
    print(f"\nGenerating {sample_size:,} random pairs for comparison...")

    random_similarities = []

    # Get list of valid NCF indices (those with embeddings)
    valid_user_ncf_ids = np.where(~user_missing_mask)[0]
    valid_sub_ncf_idxs = np.where(~sub_missing_mask)[0]

    print(f"  Valid users for random sampling: {len(valid_user_ncf_ids):,}")
    print(f"  Valid subreddits for random sampling: {len(valid_sub_ncf_idxs):,}")

    for _ in tqdm(range(sample_size), desc="Computing random pair similarities"):
        # Sample from valid NCF indices only
        rand_ncf_user = np.random.choice(valid_user_ncf_ids)
        rand_ncf_sub = np.random.choice(valid_sub_ncf_idxs)

        # Get embedding rows via reindex
        user_emb_row = user_reindex[rand_ncf_user]
        sub_emb_row = sub_reindex[rand_ncf_sub]

        sim = np.dot(user_text_normalized[user_emb_row], sub_text_normalized[sub_emb_row])
        if not np.isnan(sim):
            random_similarities.append(sim)

    random_similarities = np.array(random_similarities)

    print(f"\nRandom pair similarity stats:")
    print(f"  Mean: {random_similarities.mean():.4f}")
    print(f"  Std: {random_similarities.std():.4f}")
    print(f"  Min: {random_similarities.min():.4f}")
    print(f"  Max: {random_similarities.max():.4f}")

    # Statistical test
    from scipy import stats
    t_stat, p_value = stats.ttest_ind(positive_similarities, random_similarities)

    print(f"\nStatistical Test (t-test):")
    print(f"  t-statistic: {t_stat:.4f}")
    print(f"  p-value: {p_value:.2e}")

    diff = positive_similarities.mean() - random_similarities.mean()
    print(f"\n  Difference in means: {diff:.4f}")

    if p_value < 0.05 and diff > 0:
        print("  CONCLUSION: Positive pairs have SIGNIFICANTLY HIGHER text similarity")
        print("  This suggests text embeddings DO capture engagement signal!")
    elif p_value < 0.05 and diff < 0:
        print("  CONCLUSION: Positive pairs have SIGNIFICANTLY LOWER text similarity")
        print("  This is unexpected and suggests potential issues with embeddings.")
    else:
        print("  CONCLUSION: No significant difference in text similarity")
        print("  Text embeddings may not be capturing engagement patterns well.")

    embedding_diagnostics['text_similarity_analysis'] = {
        'positive_mean': float(positive_similarities.mean()),
        'positive_std': float(positive_similarities.std()),
        'random_mean': float(random_similarities.mean()),
        'random_std': float(random_similarities.std()),
        't_statistic': float(t_stat),
        'p_value': float(p_value),
        'mean_difference': float(diff),
        'valid_positive_pairs': valid_pairs,
        'skipped_user_missing': skipped_user,
        'skipped_sub_missing': skipped_sub
    }
else:
    print("Text embeddings not available for similarity analysis")

In [ ]:
# Visualize text similarity analysis
if 'positive_similarities' in dir() and len(positive_similarities) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Histogram comparison
    ax = axes[0]
    ax.hist(positive_similarities, bins=50, alpha=0.7, label='Positive Pairs', density=True)
    ax.hist(random_similarities, bins=50, alpha=0.7, label='Random Pairs', density=True)
    ax.axvline(positive_similarities.mean(), color='blue', linestyle='--', linewidth=2, label=f'Pos Mean: {positive_similarities.mean():.3f}')
    ax.axvline(random_similarities.mean(), color='orange', linestyle='--', linewidth=2, label=f'Rand Mean: {random_similarities.mean():.3f}')
    ax.set_xlabel('Cosine Similarity')
    ax.set_ylabel('Density')
    ax.set_title('Text Embedding Similarity: Positive vs Random Pairs')
    ax.legend()

    # Box plot comparison
    ax = axes[1]
    bp = ax.boxplot([positive_similarities, random_similarities],
                    labels=['Positive Pairs', 'Random Pairs'],
                    patch_artist=True)
    bp['boxes'][0].set_facecolor('lightblue')
    bp['boxes'][1].set_facecolor('moccasin')
    ax.set_ylabel('Cosine Similarity')
    ax.set_title('Text Embedding Similarity Distribution')

    # Add significance annotation
    if p_value < 0.001:
        sig_text = '***'
    elif p_value < 0.01:
        sig_text = '**'
    elif p_value < 0.05:
        sig_text = '*'
    else:
        sig_text = 'n.s.'

    ax.annotate(sig_text, xy=(1.5, max(positive_similarities.max(), random_similarities.max())),
                fontsize=14, ha='center')

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'text_similarity_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(f"Saved to {OUTPUT_DIR / 'text_similarity_analysis.png'}")

## 10.5 CKA Analysis: Text vs NCF Embeddings

In [ ]:
# ============================================================
# CKA ANALYSIS
# ============================================================

print("\n" + "="*70)
print("TEXT vs NCF EMBEDDING CORRELATION ANALYSIS")
print("="*70)

def linear_CKA(X, Y):
    """
    Compute linear CKA between two sets of representations.
    X: (n, d1) matrix
    Y: (n, d2) matrix
    Returns: CKA similarity score (0-1)
    """
    # Center the features
    X = X - X.mean(axis=0)
    Y = Y - Y.mean(axis=0)

    # Compute Gram matrices
    K = X @ X.T
    L = Y @ Y.T

    # Compute CKA
    hsic_kl = np.sum(K * L)
    hsic_kk = np.sum(K * K)
    hsic_ll = np.sum(L * L)

    cka = hsic_kl / (np.sqrt(hsic_kk * hsic_ll) + 1e-10)
    return cka

if state_dict is not None and sub_text_emb is not None:
    # Get NCF subreddit embeddings from model
    sub_ncf_gmf = state_dict.get('sub_embedding_gmf.weight', None)

    if sub_ncf_gmf is not None:
        sub_ncf_gmf = sub_ncf_gmf.numpy()

        print(f"\nNCF GMF Subreddit Embeddings: {sub_ncf_gmf.shape}")
        print(f"Text Subreddit Embeddings: {sub_text_emb.shape}")

        # For each NCF index, get the corresponding text embedding row

        # Find NCF indices that have valid text embeddings
        valid_ncf_idxs = np.where(~sub_missing_mask)[0]
        print(f"\nValid subreddits with both NCF and text embeddings: {len(valid_ncf_idxs)}")

        # Get aligned embeddings
        ncf_aligned = sub_ncf_gmf[valid_ncf_idxs]  # NCF embeddings for valid indices

        # Get corresponding text embedding rows
        text_emb_rows = sub_reindex[valid_ncf_idxs]
        text_aligned = sub_text_emb[text_emb_rows]  # Text embeddings at correct rows

        print(f"Aligned NCF embeddings shape: {ncf_aligned.shape}")
        print(f"Aligned Text embeddings shape: {text_aligned.shape}")

        # Handle NaN in text embeddings
        text_aligned_clean = np.nan_to_num(text_aligned, nan=0.0)

        # Compute CKA
        cka_text_gmf = linear_CKA(text_aligned_clean, ncf_aligned)
        print(f"\nCKA (Text vs NCF GMF): {cka_text_gmf:.4f}")

        # Also check NCF MLP embeddings if available
        sub_ncf_mlp = state_dict.get('sub_embedding_mlp.weight', None)
        if sub_ncf_mlp is not None:
            sub_ncf_mlp = sub_ncf_mlp.numpy()
            mlp_aligned = sub_ncf_mlp[valid_ncf_idxs]

            cka_text_mlp = linear_CKA(text_aligned_clean, mlp_aligned)
            cka_gmf_mlp = linear_CKA(ncf_aligned, mlp_aligned)

            print(f"CKA (Text vs NCF MLP): {cka_text_mlp:.4f}")
            print(f"CKA (NCF GMF vs NCF MLP): {cka_gmf_mlp:.4f}")
        else:
            cka_text_mlp = None
            cka_gmf_mlp = None

        # Interpretation
        print("\nInterpretation:")
        print("  - CKA near 1.0: Embeddings capture similar information (redundant)")
        print("  - CKA near 0.0: Embeddings capture orthogonal information (complementary)")

        if cka_text_gmf > 0.5:
            print(f"\n  WARNING: Text embeddings are {cka_text_gmf*100:.1f}% similar to NCF GMF")
            print("  This suggests text may not provide much additional information!")
        elif cka_text_gmf < 0.1:
            print(f"\n  NOTE: Text embeddings are only {cka_text_gmf*100:.1f}% similar to NCF GMF")
            print("  They capture very different information - but is it useful?")
        else:
            print(f"\n  Text embeddings are {cka_text_gmf*100:.1f}% similar to NCF GMF")
            print("  Moderate overlap - text could provide some complementary information.")

        embedding_diagnostics['cka_analysis'] = {
            'text_vs_ncf_gmf': float(cka_text_gmf),
            'text_vs_ncf_mlp': float(cka_text_mlp) if cka_text_mlp is not None else None,
            'ncf_gmf_vs_mlp': float(cka_gmf_mlp) if cka_gmf_mlp is not None else None,
            'n_aligned_subreddits': len(valid_ncf_idxs)
        }
else:
    print("Required data not available for CKA analysis")

## 10.6 Projection Layer Analysis

In [ ]:
# Analyze projection layer and embedding magnitudes
if state_dict is not None and user_text_emb is not None:
    print("\n" + "="*60)
    print("PROJECTION LAYER AND MAGNITUDE ANALYSIS")
    print("="*60)

    # User text projection
    if 'user_text_proj.weight' in state_dict:
        user_proj_weight = state_dict['user_text_proj.weight'].numpy()
        user_proj_bias = state_dict.get('user_text_proj.bias', torch.zeros(user_proj_weight.shape[0])).numpy()

        print("\nUser Text Projection Layer:")
        print(f"  Weight shape: {user_proj_weight.shape}")
        print(f"  Weight mean: {user_proj_weight.mean():.6f}")
        print(f"  Weight std: {user_proj_weight.std():.6f}")

        # Apply projection to sample of user text embeddings (using correct indexing)
        valid_user_ncf_ids = np.where(~user_missing_mask)[0][:1000]  # Sample 1000
        valid_emb_rows = user_reindex[valid_user_ncf_ids]

        sample_text_emb = np.nan_to_num(user_text_emb[valid_emb_rows], nan=0.0)

        # Project
        projected = sample_text_emb @ user_proj_weight.T + user_proj_bias

        # Compare norms before/after
        norms_before = np.linalg.norm(sample_text_emb, axis=1)
        norms_after = np.linalg.norm(projected, axis=1)

        print(f"\n  Norm before projection: {norms_before.mean():.4f} (+/- {norms_before.std():.4f})")
        print(f"  Norm after projection: {norms_after.mean():.4f} (+/- {norms_after.std():.4f})")
        print(f"  Norm ratio (after/before): {(norms_after.mean() / norms_before.mean()):.4f}")

        embedding_diagnostics['projection_analysis'] = {
            'user_norm_before': float(norms_before.mean()),
            'user_norm_after': float(norms_after.mean()),
            'user_norm_ratio': float(norms_after.mean() / norms_before.mean())
        }

    # Compare to NCF embedding norms
    user_ncf_gmf = state_dict.get('user_embedding_gmf.weight', None)
    if user_ncf_gmf is not None:
        user_ncf_gmf = user_ncf_gmf.numpy()
        ncf_norms = np.linalg.norm(user_ncf_gmf, axis=1)

        print(f"\n  NCF GMF embedding norm: {ncf_norms.mean():.4f} (+/- {ncf_norms.std():.4f})")

        if 'projection_analysis' in embedding_diagnostics:
            proj_norm = embedding_diagnostics['projection_analysis']['user_norm_after']
            print(f"  Projected text / NCF ratio: {proj_norm / ncf_norms.mean():.4f}")

            if abs(proj_norm / ncf_norms.mean() - 1) > 0.5:
                print("  WARNING: Significant scale mismatch between projected text and NCF embeddings!")

## 11. Save Results

In [ ]:
# Save all diagnostics
print("\n" + "="*70)
print("SAVING ALL RESULTS")
print("="*70)

with open(OUTPUT_DIR / 'embedding_diagnostics.json', 'w') as f:
    json.dump(embedding_diagnostics, f, indent=2)

print(f"Saved diagnostics to {OUTPUT_DIR / 'embedding_diagnostics.json'}")

# Save Venn stats
with open(OUTPUT_DIR / 'venn_stats.json', 'w') as f:
    json.dump(venn_stats, f, indent=2)

print(f"Saved Venn stats to {OUTPUT_DIR / 'venn_stats.json'}")

In [ ]:
# Print final summary
print("\n" + "="*70)
print("ANALYSIS SUMMARY")
print("="*70)

print("\n1. INDEX MAPPING VERIFICATION:")
print(f"   User coverage: {(~user_missing_mask).sum()}/{num_users} ({(~user_missing_mask).sum()/num_users*100:.1f}%)")
print(f"   Subreddit coverage: {(~sub_missing_mask).sum()}/{num_subreddits} ({(~sub_missing_mask).sum()/num_subreddits*100:.1f}%)")

print("\n2. OVERALL PERFORMANCE:")
for model in ['cosine_4096', 'ncf_baseline', 'ncf_text']:
    hit_rate = hits_df[f'hit_{model}'].mean()
    print(f"   {model}: HR@10 = {hit_rate:.4f} ({hit_rate*100:.2f}%)")

print("\n3. MODEL OVERLAP (Venn Diagram):")
print(f"   NCF Baseline vs NCF+Text overlap: {venn_stats['ncf_baseline_and_ncf_text'] + venn_stats['all_three']:,} pairs")
print(f"   Pairs hit ONLY by NCF+Text: {venn_stats['only_ncf_text']:,}")
print(f"   Pairs hit ONLY by NCF Baseline: {venn_stats['only_ncf_baseline']:,}")

print("\n4. EMBEDDING DIAGNOSTICS:")
if 'text_similarity_analysis' in embedding_diagnostics:
    tsa = embedding_diagnostics['text_similarity_analysis']
    print(f"   Positive pair similarity: {tsa['positive_mean']:.4f} (+/- {tsa['positive_std']:.4f})")
    print(f"   Random pair similarity: {tsa['random_mean']:.4f} (+/- {tsa['random_std']:.4f})")
    print(f"   Difference: {tsa['mean_difference']:.4f} (p={tsa['p_value']:.2e})")
    print(f"   Valid pairs analyzed: {tsa['valid_positive_pairs']:,}")

if 'cka_analysis' in embedding_diagnostics:
    cka = embedding_diagnostics['cka_analysis']
    print(f"\n   CKA (Text vs NCF GMF): {cka['text_vs_ncf_gmf']:.4f}")
    print(f"   Aligned subreddits: {cka['n_aligned_subreddits']}")

print("\n5. KEY FINDINGS:")
if venn_stats['only_ncf_text'] < venn_stats['only_ncf_baseline']:
    print("   - NCF Baseline captures MORE unique pairs than NCF+Text")
    print("   - Text embeddings may be adding noise rather than signal")
else:
    print("   - NCF+Text captures MORE unique pairs than NCF Baseline")

print("\n6. OUTPUTS GENERATED:")
print(f"   Directory: {OUTPUT_DIR}")
for f in sorted(os.listdir(OUTPUT_DIR)):
    print(f"   - {f}")

## Appendix: Verify Index Alignment is Correct

This cell provides additional verification that our reindexing is working correctly.

In [ ]:
# Additional verification of index alignment
print("="*70)
print("INDEX ALIGNMENT VERIFICATION")
print("="*70)

print("\n1. USER INDEX VERIFICATION:")
print(f"   user_ids.json contains {len(embedding_user_ids)} entries")
print(f"   NCF has {num_users} users")

# Check if user_ids.json is sorted (which would mean row i = NCF user i)
is_sorted = embedding_user_ids == sorted(embedding_user_ids)
is_sequential = embedding_user_ids == list(range(len(embedding_user_ids)))

print(f"   user_ids.json is sorted: {is_sorted}")
print(f"   user_ids.json is sequential [0,1,2,...]: {is_sequential}")

if is_sequential:
    print("   -> User embeddings ARE directly aligned with NCF indices!")
else:
    print("   -> User embeddings require reindexing (our fix handles this)")

print("\n2. SUBREDDIT INDEX VERIFICATION:")
print(f"   subreddit_names.json contains {len(embedding_sub_names)} entries")
print(f"   NCF has {num_subreddits} subreddits")
print(f"   First 5 embedding names: {embedding_sub_names[:5]}")
print(f"   First 5 NCF subreddits: {subreddit_names[:5]}")

# Check if order matches
order_matches = embedding_sub_names[:min(10, len(embedding_sub_names))] == subreddit_names[:min(10, len(subreddit_names))]
print(f"   Order matches (first 10): {order_matches}")

if not order_matches:
    print("   -> Subreddit embeddings require reindexing (our fix handles this)")

print("\n3. SPOT CHECK:")
# Pick a random NCF subreddit index and verify the mapping
test_ncf_idx = 42
test_sub_name = idx2subreddit.get(test_ncf_idx, 'UNKNOWN')
test_emb_row = sub_reindex[test_ncf_idx]
if test_emb_row >= 0:
    recovered_name = embedding_sub_names[test_emb_row]
    print(f"   NCF idx {test_ncf_idx} = '{test_sub_name}'")
    print(f"   -> maps to embedding row {test_emb_row}")
    print(f"   -> embedding row {test_emb_row} = '{recovered_name}'")
    print(f"   -> Match: {test_sub_name == recovered_name}")